# 500줄로 ClaudeCode 만들기

이 노트북은 LLM 기반 코드 편집 에이전트를 0단계부터 5단계까지 확장해 가며 따라가는 1강 자료입니다.

이번 강의는 loop-agent를 처음 구현하는 과정입니다. 예제만 실행해 보고 넘어가기보다는, AI 도움 없이 최종 결과물을 손수 코딩해 보는 경험을 권장합니다. 막히는 지점이 생긴 뒤에 코드를 비교하면 conversation, tool calling, tool feedback loop가 훨씬 분명하게 보입니다.

진행 방식은 단순합니다.

1. 이 노트북을 저장소 루트에서 엽니다.
2. 실행 셀을 클릭하면 새 macOS Terminal.app 창이 열립니다.
3. `복사할 입력` 셀의 `text` 블록 복사 버튼을 눌러 터미널에 붙여넣습니다.
4. 프로그램을 멈출 때는 터미널에서 `Ctrl-C`를 누릅니다.

학생이 직접 입력할 것은 `복사할 입력`으로 분리된 `text` 블록뿐입니다. 터미널에 칠 명령은 모두 실행 셀로 만들어 두었습니다.

## 0. 준비

의존성을 설치합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv sync

`.env` 파일이 없으면 `.env.example`에서 만듭니다. 이미 `.env`가 있으면 덮어쓰지 않습니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && (test -f .env || cp .env.example .env)

`.env`에는 OpenRouter 또는 Gemini API 키를 넣습니다. 실행 셀은 실제 API를 호출하므로 유효한 키가 없으면 실패합니다.

테스트는 실제 LLM API를 호출하지 않습니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run --group dev pytest tests

## 전체 발전 흐름

| 단계 | 실행 모듈 | 발전 포인트 | 핵심 한계 |
| --- | --- | --- | --- |
| 0 | `step_00_no_memory` | 현재 입력 하나만 보내 LLM이 요청 사이를 기억하지 못함을 확인 | 이전 대화를 기억할 수 없음 |
| 1 | `step_01_basic` | conversation을 누적해 대화 기억 추가 | 파일을 읽거나 실행할 수 없음 |
| 2 | `step_02_tool` | `read_file` tool calling 첫 도입 | 도구 결과를 LLM에게 다시 전달하지 않음 |
| 3 | `step_03_tool_loop` | tool 결과를 `role="tool"` 메시지로 돌려줌 | 읽기 도구만 있어서 수정/실행 불가 |
| 4 | `step_04_tool_extend` | `list_dir`, `edit_file`로 탐색/생성/수정 추가 | 작업공간 보호와 실행 검증이 약함 |
| 5 | `step_05_code_agent` | 경로 보호, `write_file`, 정확한 `edit_file`, `run_node_file` 추가 | JavaScript 실행만 지원 |

## 1. 0단계: 기억 없는 LLM 호출

아래 셀을 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_00_no_memory

복사할 입력 1:

응답을 확인한 뒤, 같은 터미널에 이어서 붙여넣습니다.

복사할 입력 2:

관찰 포인트:

- 0단계는 두 번째 요청에 첫 번째 메시지를 함께 보내지 않습니다.
- 모델이 이름을 맞힌다면 구조적 기억이 아니라 추측입니다.
- LLM API는 요청 사이를 자동으로 기억하지 않습니다.

## 2. 1단계: conversation으로 대화 기억 추가

아래 셀을 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_01_basic

복사할 입력 1:

응답을 확인한 뒤 이어서 붙여넣습니다.

복사할 입력 2:

관찰 포인트:

- 1단계는 사용자 입력과 모델 응답을 `conversation` 리스트에 누적합니다.
- 두 번째 요청에는 첫 번째 사용자 메시지와 모델 응답이 함께 들어갑니다.
- 기억은 모델 내부에 저장되는 것이 아니라, 우리가 conversation을 다시 보내기 때문에 생깁니다.

## 3. 1단계 보조 실험: 프롬프트만으로 함수 호출 흉내 내기

같은 1단계 프로그램을 다시 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_01_basic

복사할 입력 1:

응답을 확인한 뒤 이어서 붙여넣습니다.

복사할 입력 2:

관찰 포인트:

- 모델은 `get_weather(Seoul)` 같은 텍스트를 만들 수 있습니다.
- 하지만 실제 `get_weather` 함수가 실행되는 것은 아닙니다.
- 2단계부터는 이 한계를 해결하기 위해 진짜 tool calling을 추가합니다.

## 4. 2단계: 파일 읽기 도구 추가

아래 셀을 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_02_tool

복사할 입력:

관찰 포인트:

- 모델이 `read_file` 도구 호출을 만들 수 있습니다.
- 프로그램은 실제로 파일을 읽습니다.
- 하지만 이 단계에서는 파일 내용이 다시 모델에게 전달되지 않습니다.
- 그래서 답변이 완성되지 않거나 어색할 수 있습니다.

## 5. 3단계: Tool Feedback Loop

아래 셀을 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_03_tool_loop

복사할 입력:

관찰 포인트:

- 3단계는 도구 실행 결과를 `role="tool"` 메시지로 conversation에 추가합니다.
- 그 뒤 사용자 입력 없이 모델을 한 번 더 호출합니다.
- 이 구조가 있어야 모델이 파일 내용을 읽은 뒤 그 내용을 바탕으로 답변할 수 있습니다.

## 6. 3단계 보조 실험: 코드 읽고 설명하기

같은 3단계 프로그램을 다시 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_03_tool_loop

복사할 입력:

관찰 포인트:

- 모델은 먼저 `read_file` 도구로 파일을 읽어야 합니다.
- 그 뒤 tool 결과를 바탕으로 간단한 설명을 생성합니다.

## 7. 4단계: 파일 도구 확장

아래 셀을 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_04_tool_extend

제공 도구:

- `read_file`: 파일 내용 읽기
- `list_dir`: 디렉터리 목록 보기
- `edit_file`: 파일 수정
- `create_new_file`: `edit_file` 내부에서 새 파일 생성 시 사용하는 헬퍼

복사할 입력 1:

복사할 입력 2:

복사할 입력 3:

복사할 입력 4:

생성된 JavaScript를 직접 실행하려면 아래 셀을 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py node fizzbuzz.js

관찰 포인트:

- 이제 모델이 파일을 탐색하고 만들고 수정할 수 있습니다.
- 하지만 작업공간 밖 경로 접근 보호가 약합니다.
- Node.js 실행 검증은 사용자가 직접 해야 합니다.

## 8. 5단계: 코드 편집 에이전트 완성형

아래 셀을 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_05_code_agent

제공 도구:

- `read_file`: 작업공간 내부 파일 읽기
- `list_dir`: 작업공간 내부 디렉터리 목록 보기
- `write_file`: 파일 전체 내용 생성 또는 덮어쓰기
- `edit_file`: 기존 파일의 특정 문자열을 정확히 한 번만 교체
- `run_node_file`: JavaScript 파일을 Node.js로 실행하고 결과 반환
- `resolve_workspace_path`: 작업공간 밖 경로 접근 방지

복사할 입력 1:

복사할 입력 2:

파일 생성이 끝나면 터미널에서 `Ctrl-C`로 종료합니다. 그다음 아래 셀을 클릭해 새 터미널에서 직접 실행해 봅니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py node fizzbuzz.js

다시 5단계를 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_05_code_agent

복사할 입력 3:

관찰 포인트:

- 새 파일 생성은 `write_file`이 담당합니다.
- 기존 파일 수정은 `edit_file`이 담당합니다.
- JavaScript 실행 검증은 `run_node_file`이 담당합니다.

## 9. 작업공간 보호 확인

5단계를 다시 실행합니다.

In [ ]:
!ROOT=$(git rev-parse --show-toplevel) && cd "$ROOT" && uv run python scripts/open_terminal.py uv run python -m lecture_001_500_lines_claudecode.steps.step_05_code_agent

복사할 입력:

관찰 포인트:

- 5단계는 현재 작업 디렉터리 밖의 파일을 읽거나 쓸 수 없도록 막아야 합니다.
- `resolve_workspace_path`가 이 제한을 담당합니다.

## 10. 코드 구조 핵심 개념

### Conversation

LLM은 이전 요청을 자동으로 기억하지 않습니다. 채팅처럼 보이게 하려면 이전 사용자 메시지와 모델 응답을 conversation에 누적하고 매 요청마다 다시 보내야 합니다.

### ToolDefinition

`ToolDefinition`은 도구 이름, 설명, JSON Schema 입력, 실제 Python 함수를 하나로 묶습니다. 모델은 schema를 보고 tool call을 만들고, 로컬 프로그램은 해당 Python 함수를 실행합니다.

### Tool Feedback Loop

도구 실행 결과를 `role="tool"` 메시지로 conversation에 추가해야 모델이 그 결과를 읽고 최종 답변을 만들 수 있습니다.

### Workspace Guard

5단계의 `resolve_workspace_path`는 `../secret.txt`처럼 작업공간 밖으로 나가려는 경로를 차단합니다. 코드 편집 에이전트에서 가장 중요한 안전장치 중 하나입니다.

## 11. 정리

추천 학습 순서:

1. 먼저 각 단계 파일을 열어 코드를 훑습니다.
2. 노트북의 실행 셀을 직접 실행하고, 안내된 프롬프트를 붙여넣습니다.
3. 출력이 왜 그렇게 나왔는지 바로 위 단계와 비교합니다.
4. 마지막으로 5단계 결과물을 AI 도움 없이 직접 다시 작성해 봅니다.
5. 막히는 부분만 원본 코드와 비교합니다.

실습 중 생성되는 `fizzbuzz.js` 같은 파일은 필요 없으면 삭제해도 됩니다.